In [1]:
pip install openai langchain faiss-cpu

Note: you may need to restart the kernel to use updated packages.


Base Agent Class

In [2]:
# agent_base.py
from abc import ABC, abstractmethod

class Agent(ABC):
    """Base class for modular agents."""
    
    @abstractmethod
    def run(self, input_data: dict) -> dict:
        pass

Retriever Agent (GraphRAG-style retrieval)

In [3]:
# retriever_agent.py
from langchain.vectorstores import FAISS

class RetrieverAgent(Agent):
    def __init__(self, vectorstore: FAISS):
        self.vectorstore = vectorstore

    def run(self, input_data: dict) -> dict:
        query = input_data.get("query")
        results = self.vectorstore.similarity_search(query, k=5)
        return {"retrieved_docs": results}


Enrichment Agent (OpenAI LLM)

In [4]:
# enrichment_agent.py
from openai import OpenAI

client = OpenAI()

class EnrichmentAgent(Agent):
    def run(self, input_data: dict) -> dict:
        docs = input_data.get("retrieved_docs", [])
        enriched = []
        for doc in docs:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role":"system","content":"You are a captioning agent."},
                          {"role":"user","content":f"Generate a dense caption for: {doc.page_content}"}]
            )
            enriched.append(response.choices[0].message.content)
        return {"enriched_docs": enriched}

Synthesizer Agent (LLM Summarization)

In [5]:
# synthesizer_agent.py
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS

client = OpenAI()

class SynthesizerAgent(Agent):
    def run(self, input_data: dict) -> dict:
        enriched = input_data.get("enriched_docs", [])
        joined = "\n".join(enriched)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role":"system","content":"You are a synthesis agent."},
                      {"role":"user","content":f"Synthesize the following into a structured summary:\n{joined}"}]
        )
        return {"summary": response.choices[0].message.content}

GraphRAG Orchestrator (Parallel Branches)

In [6]:
# orchestrator.py
from concurrent.futures import ThreadPoolExecutor

class GraphOrchestrator:
    def __init__(self, retrievers, enrichers, synthesizer):
        self.retrievers = retrievers
        self.enrichers = enrichers
        self.synthesizer = synthesizer

    def execute(self, query: str) -> dict:
        # Parallel retrieval
        with ThreadPoolExecutor() as executor:
            retrieval_results = list(executor.map(
                lambda r: r.run({"query": query}), self.retrievers
            ))

        # Parallel enrichment
        with ThreadPoolExecutor() as executor:
            enrichment_results = list(executor.map(
                lambda e, docs: e.run(docs),
                self.enrichers,
                retrieval_results
            ))

        # Merge results
        merged_docs = []
        for branch in enrichment_results:
            merged_docs.extend(branch.get("enriched_docs", []))

        # Synthesis
        summary = self.synthesizer.run({"enriched_docs": merged_docs})
        return summary

In [7]:
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document

# Initialize OpenAI embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [8]:
# Example documents
docs = [
    Document(page_content="Multi-agent systems coordinate autonomous agents."),
    Document(page_content="GraphRAG enables branching retrieval and synthesis."),
    Document(page_content="Dense captioning enriches retrieved content with CV+LLM.")
]

# Semantic store (FAISS index)
semantic_store = FAISS.from_documents(docs, embeddings)

# Graph store (could be another FAISS index, or a different retrieval strategy)
graph_store = FAISS.from_documents(docs, embeddings)


In [9]:
# main_notebook.ipynb
retriever_semantic = RetrieverAgent(vectorstore=semantic_store)
retriever_graph = RetrieverAgent(vectorstore=graph_store)

enrichment_dense = EnrichmentAgent()
enrichment_cvllm = EnrichmentAgent()

synthesizer = SynthesizerAgent()

workflow = GraphOrchestrator(
    retrievers=[retriever_semantic, retriever_graph],
    enrichers=[enrichment_dense, enrichment_cvllm],
    synthesizer=synthesizer
)

query = "Explain modular multi-agent workflows"
result = workflow.execute(query)

print("Final Summary:\n", result["summary"])


Final Summary:
 ### Structured Summary

**1. Multi-Agent Systems**  
- Definition: Multi-agent systems consist of autonomous agents working together.  
- Purpose: They facilitate coordination and collaboration to achieve shared objectives efficiently.  
- Features: Enable coordinated interactions to accomplish complex tasks.

**2. GraphRAG**  
- Functionality: A method that enhances information retrieval processes by utilizing structured graph-based techniques.  
- Innovations: Introduces advanced branching retrieval and synthesis, revolutionizing how data is accessed and organized.

**3. Dense Captioning**  
- Integration: Combines computer vision (CV) with large language models (LLM).  
- Purpose: Aims to enrich visual data with context-aware textual descriptions.  
- Outcome: Provides nuanced and informative insights into retrieved content, enhancing understanding of visual information.
